# R3-M1 batch 02: hidden size 64 (experimental only)
One fresh two-layer BiGRU fit: hidden size 64, LR 0.0003, seed 42, max 20 epochs. Compare against four preserved candidates. Results are advisory and cannot change locked results. No calibration, operating, test or Chatterbox scoring.

This notebook requires explicit Muneeb approval recorded in `configs/batch02_authorization.json` for this plan and the current locks. Do not create this approval yourself, delete lock files, or edit hashes to bypass a refusal. Preserve the exact batch01 runtime versions and files. Use one T4 session.


In [11]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')
EXP = ROOT / 'experiments/r3_m1'
SHARDS = ROOT / 'data/processed/cohort_v1'
CONTRACT = EXP / 'configs/batch02_resolved.json'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
TEMP = EXP / 'runs/batch02/notebook_tmp'
TEMP.mkdir(parents=True, exist_ok=True)
for key in ('TMPDIR', 'TEMP', 'TMP'):
    os.environ[key] = str(TEMP)
assert (EXP / 'configs/batch02_authorization.json').is_file(), 'Muneeb approval is required before this post-lock diagnostic'
assert SHARDS.is_dir()
def run_script(name, *args):
    subprocess.run([sys.executable, str(EXP / 'scripts' / name), *map(str, args)], check=True)


## Freeze, verify, and check fixtures
A contract is created once. It pins inputs, source code, versions, architecture, all prior candidates, authorization, and existing locked artifacts. Reconnecting reuses it. Full fixture checks must pass without skips, including hidden64 CPU and CUDA gradient/padding/reload checks. Check time is counted against a shared 15-minute allowance.


In [13]:
if not CONTRACT.exists():
    run_script('batch02_contract.py')
else:
    run_script('batch02_contract.py', '--verify')
preflight = EXP / 'reports/batch02/preflight.json'
if not preflight.exists():
    run_script('batch02_preflight.py')
else:
    print('Existing fixture evidence will be verified by the runner:', preflight)


## One bounded fit
One candidate, at most 20 epochs and 60 cumulative training/validation minutes. An existing run resumes only when its checkpoints and epoch outputs are consistent; a completed run is reused. Do not remove an active-attempt marker after a hard disconnect. Resume has the existing CUDA GRU dropout reproducibility limitation.


In [14]:
run_id = 'b02_hidden64_full20'
directory = EXP / 'runs/batch02' / run_id
args = ['--contract', CONTRACT, '--experiment', run_id, '--shard-dir', SHARDS]
if directory.exists():
    args += ['--resume']
run_script('batch02_train.py', *args)


## Advisory comparison only
Recompute all five candidates from identical selection recording/chunk identities. The ranking uses full-selection macro F1 at 0.5, with the existing tie rules. XTTS recall, false positives, AUC and early-stop trajectories are diagnostics. Existing official and batch01 decisions remain unchanged. There is no calibration/operating export cell.


In [15]:
run_script('batch02_compare.py', '--contract', CONTRACT)
print((EXP / 'reports/batch02/comparison.md').read_text())
print('Copy back: configs/batch02_resolved.json, reports/batch02/, and runs/batch02/.')
print('The team must review any proposal; this notebook does not promote or deploy a model.')


# Batch 02: hidden-size-64 comparison

| Candidate | Hidden size | Best epoch | Full macro F1 | XTTS recall | Genuine FPR | XTTS AUC |
|---|---:|---:|---:|---:|---:|---:|
| original_baseline | 128 | 3 | 0.859754 | 0.926000 | 0.310000 | 0.884704 |
| original_lower_lr | 128 | 10 | 0.861128 | 0.676000 | 0.118000 | 0.883184 |
| b01_baseline_full20 | 128 | 7 | 0.867587 | 0.702000 | 0.120000 | 0.882236 |
| b01_lower_lr_full20 | 128 | 16 | 0.869120 | 0.664000 | 0.084000 | 0.904064 |
| b02_hidden64_full20 | 64 | 10 | 0.838414 | 0.604000 | 0.126000 | 0.841564 |

Experimental preference: b01_lower_lr_full20. Advisory only; team review pending.

Isolated post-lock, result-informed single-seed diagnostic. Advisory only; no automatic promotion, calibration/operating evaluation, statistical superiority, or unseen-generator guarantee.

Choice follows full selection macro F1 at 0.5. XTTS diagnostics do not override that rule.

Hidden64 F1 difference from the batch01 selected checkpoint: -0.030706.

Co